# AutoML para classificação binária

Comparação entre FLAML e AutoGluon no dataset público **Breast Cancer Wisconsin (Diagnostic)**. Notebooks preparados a partir de uma atividade acadêmica. Este experimento não é uma aplicação de diagnóstico clínico.

Execute o Jupyter Notebook a partir da pasta raiz do repositório após instalar as dependências do `requirements.txt`.

## 1. Inspeção dos dados

A tabela original possui 569 linhas, 30 variáveis preditoras, a coluna `diagnosis` e campos auxiliares (`id`, `Unnamed: 32`). Esses campos auxiliares são descartados antes do treino.

In [ ]:
import pandas as pd

# 1. Carregando o dataset
# (Certifique-se de fazer o upload do arquivo 'dataset_cancer.csv' no Colab)
df = pd.read_csv('dados/dataset_cancer.csv')

# 2. Comprovando a quantidade de registros (linhas) e atributos (colunas)
print("--- DIMENSÕES DO DATASET ---")
linhas, colunas = df.shape
print(f"Registros (linhas): {linhas}")
print(f"Atributos (colunas): {colunas}\n")

# 3. Comprovando a distribuição da variável-alvo (Dados desbalanceados)
print("--- DISTRIBUIÇÃO DA VARIÁVEL-ALVO (diagnosis) ---")
distribuicao = df['diagnosis'].value_counts()
print(distribuicao)
print(f"Porcentagem Benigno (B): {(distribuicao['B'] / linhas) * 100:.2f}%")
print(f"Porcentagem Maligno (M): {(distribuicao['M'] / linhas) * 100:.2f}%\n")

# 4. Comprovando os valores ausentes (Missing Values)
print("--- VALORES AUSENTES POR COLUNA ---")
valores_ausentes = df.isnull().sum()
# Mostra apenas as colunas que têm valores nulos
print(valores_ausentes[valores_ausentes > 0]) 

# 5. Comprovando os tipos de dados (Atributos categóricos vs numéricos)
print("\n--- TIPOS DOS ATRIBUTOS ---")
print(df.dtypes)

## 2. FLAML

Treinamento com `max_iter=50`, métrica `accuracy` e divisão aleatória de 80% para treino e 20% para teste (`random_state=123`). Sem estratificação nesta versão original.

In [ ]:
import time
from pathlib import Path

Path("logs").mkdir(exist_ok=True)
from flaml import AutoML
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# 1. Carregar e limpar os dados
print("[1/4] A carregar e limpar os dados...")
df = pd.read_csv('dados/dataset_cancer.csv')
df = df.drop(columns=['id', 'Unnamed: 32'], errors='ignore')

# 2. Separar os dados em X (features) e y (alvo)
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']

# Dividir em treino (80%) e teste (20%) - random_state garante os mesmos dados sempre
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

# 3. Inicializar e Treinar o FLAML AutoML
print("\n[2/4] A iniciar o AutoML do FLAML (Sem limite de tempo, a testar 50 modelos)...")
automl = AutoML()

# Configurações: tarefa de classificação, máximo de 50 modelos iterados
settings = {
    "max_iter": 50,  
    "metric": 'accuracy', 
    "task": 'classification',
    "log_file_name": 'logs/flaml_log.log',
    "verbose": 0 # Deixamos em 0 para não poluir o ecrã com muito texto
}

# CRONOMETRAR O TEMPO DE TREINO
inicio_flaml = time.time()
automl.fit(X_train=X_train, y_train=y_train, **settings)
fim_flaml = time.time()

# Calcular o tempo total
tempo_total_flaml = fim_flaml - inicio_flaml

# 4. Resultados do Melhor Modelo e Tempo
print("\n[3/4] Treino concluído!")
print("\n--- RESULTADOS DO TREINO ---")
print(f"--> TEMPO DE EXECUÇÃO: {tempo_total_flaml:.2f} segundos")
print(f"--> Algoritmo vencedor: {automl.best_estimator.upper()}")
print(f"--> Melhores hiperparâmetros: {automl.best_config}")

# 5. Avaliação e Gráficos
print("\n[4/4] A gerar gráficos e métricas nos dados de Teste inéditos...")
y_pred = automl.predict(X_test)

# Relatório de Classificação (Acurácia, F1, Recall)
print("\n--- RELATÓRIO DE MÉTRICAS ---")
print(classification_report(y_test, y_pred))

# Matriz de Confusão
print("\n--- MATRIZ DE CONFUSÃO ---")
cm = confusion_matrix(y_test, y_pred, labels=automl.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=automl.classes_)
disp.plot(cmap='Blues')
plt.title(f"Matriz de Confusão - FLAML ({automl.best_estimator.upper()})")
plt.show()

# Importância dos Atributos (Feature Importance)
print("\n--- IMPORTÂNCIA DOS ATRIBUTOS ---")
plt.figure(figsize=(10, 6))
# Extrai a importância das variáveis do modelo vencedor
feat_importances = pd.Series(automl.model.estimator.feature_importances_, index=X.columns)
# Plota as 10 mais importantes ordenadas
feat_importances.nlargest(10).sort_values().plot(kind='barh', color='teal')
plt.title(f"Top 10 Atributos Mais Importantes ({automl.best_estimator.upper()})")
plt.xlabel("Grau de Importância")
plt.show()

## 3. AutoGluon

O AutoGluon utiliza a mesma divisão externa de treino/teste. Sua seleção interna de modelos utiliza validação a partir dos dados de treinamento.

In [ ]:
import time
from autogluon.tabular import TabularPredictor
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt

# 1. Carregar e limpar os dados
print("[1/5] A carregar os dados...")
df = pd.read_csv('dados/dataset_cancer.csv')
df = df.drop(columns=['id', 'Unnamed: 32'], errors='ignore')

# 2. Dividir em Treino (80%) e Teste (20%)
# Mantemos a mesma divisão treino/teste para comparar os resultados nesta amostra.
# Os procedimentos internos de seleção e validação diferem entre os frameworks.
train_data, test_data = train_test_split(df, test_size=0.2, random_state=123)

# 3. Inicializar e cronometrar o AutoGluon
print("\n[2/5] A iniciar o treino do AutoGluon (Sem limite de tempo)...")
print("O AutoGluon vai testar vários modelos e criar um Ensemble. Aguarde...")

inicio_ag = time.time()

# O AutoGluon identifica automaticamente que é uma classificação binária baseada na coluna 'diagnosis'
# Não colocamos time_limit para ele fazer o ciclo completo naturalmente
predictor = TabularPredictor(label='diagnosis', eval_metric='accuracy', path='modelos/autogluon').fit(train_data)

fim_ag = time.time()
tempo_total_ag = fim_ag - inicio_ag

# 4. Resultados e Tabela de Classificação (Leaderboard)
print(f"\n[3/5] Treino AutoGluon concluído!")
print(f"--> TEMPO DE EXECUÇÃO: {tempo_total_ag:.2f} segundos")

print("\n--- LEADERBOARD (TABELA DE CLASSIFICAÇÃO) ---")
# O Leaderboard mostra todos os modelos testados e a pontuação de cada um
leaderboard = predictor.leaderboard(test_data)
print(leaderboard)

print("\n--- MELHOR MODELO ENCONTRADO ---")
melhor_modelo = predictor.model_best
print(melhor_modelo)

# 5. Avaliação nos dados de Teste e Matriz de Confusão
print("\n[4/5] A avaliar métricas e a gerar a Matriz de Confusão...")
y_test = test_data['diagnosis']
y_pred = predictor.predict(test_data)

print("\n--- RELATÓRIO DE MÉTRICAS ---")
print(classification_report(y_test, y_pred))

print("\n--- MATRIZ DE CONFUSÃO ---")
cm = confusion_matrix(y_test, y_pred, labels=predictor.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=predictor.classes_)
# Utilizei a cor roxa (Purples) para distinguir facilmente dos gráficos azuis do FLAML no seu relatório
disp.plot(cmap='Purples') 
plt.title("Matriz de Confusão - AutoGluon")
plt.show()

# 6. Importância dos Atributos
print("\n[5/5] A calcular a Importância dos Atributos (isto pode demorar alguns segundos)...")
print("\n--- IMPORTÂNCIA DOS ATRIBUTOS ---")
feature_importance = predictor.feature_importance(test_data)

plt.figure(figsize=(10, 6))
# Plota os 10 atributos mais importantes
feature_importance['importance'].nlargest(10).sort_values().plot(kind='barh', color='indigo')
plt.title("Top 10 Atributos Mais Importantes (AutoGluon)")
plt.xlabel("Grau de Importância")
plt.show()

## Resultados observados na execução original

- **FLAML**: acurácia aproximada de **98,25%** (112 acertos em 114 exemplos).
- **AutoGluon**: acurácia aproximada de **99,12%** (113 acertos em 114 exemplos).

Os valores são de **uma única divisão dos dados**. Não demonstram superioridade estatística do AutoGluon nem validade clínica. Os gráficos desta execução foram extraídos e preservados em `imagens/`. Resultados podem variar com versões e ambiente.